# Section VI-D — Neuron-Level Causal Validation of Modality Competition
## Causal Ablation and Transfer Retention Pipeline (v2 — Clean Protocol Execution)

This notebook executes the complete 4-week experimental protocol designed by Prof. KC Lan to evaluate whether multimodal neural networks develop specialized, load-bearing neuron substrates for dominant input modalities, and whether fine-tuning preserves, reassigns, or disperses those substrates.

### Protocol Roadmap:
- **Day 0:** Setup, Path Resolution, Layer Verification, and Dual SHAP Attribution Reporting (`sum(|phi|)` vs `mean(|phi|)`).
- **Week 1 (Days 1–5):** Leakage-Free Activation Extraction & Sparse L1-Logistic Probing.
- **Week 2 (Days 6–10):** Train-Clamped Causal Mean-Ablation & Dose-Response Sweeps ($k \in \{1 \dots 64\}$).
- **Week 3 (Days 11–15):** Day 13 Cosine Similarity & Day 14 Hybrid Epsilon-Screened ($\epsilon=0.05$) Transfer Retention Taxonomy.
- **Week 4 (Days 16–21+):** Publication Tables, Dose-Response Plots, and Paper Integration.

In [1]:
# ── Environment Setup & Reproducibility Utilities ──
import os
import sys
import pickle
import numpy as np
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import cosine_similarity

# Set deterministic seed across PyTorch and NumPy
def set_deterministic_seed(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_deterministic_seed(0)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Runtime Device: {device}")

# Global Canonical Constants
EMOTION_CLASSES = ["anger", "disgust", "fear", "happiness", "sadness", "surprise"]
DEFAULT_MODALITY_SLICES = {
    "Text": (0, 1024),
    "Video": (1024, 1088),
    "Audio": (1088, 1152),
}

# Standalone Pure Mathematical Utilities
def compute_modality_attributions(shap_values, slices=None):
    if slices is None:
        slices = DEFAULT_MODALITY_SLICES
    if isinstance(shap_values, list):
        shap_array = np.vstack([np.asarray(s) for s in shap_values])
    else:
        shap_array = np.asarray(shap_values)

    results = {}
    for mod_name, (start, end) in slices.items():
        mod_shap = shap_array[:, start:end]
        feature_dim = end - start
        results[mod_name] = {
            "mean_abs_phi": float(np.mean(np.abs(mod_shap))),
            "sum_abs_phi": float(np.sum(np.abs(mod_shap))),
            "abs_sum_phi": float(np.abs(np.sum(mod_shap))),
            "feature_dim": float(feature_dim),
            "slice_start": float(start),
            "slice_end": float(end),
        }
    return results

def generate_dose_response_ks(feature_dim=64, custom_ks=None):
    if custom_ks is None:
        custom_ks = [1, 3, 5, 10, 16, 32, 48, 64]
    valid_ks = sorted({k for k in custom_ks if 1 <= k <= feature_dim})
    if feature_dim not in valid_ks:
        valid_ks.append(feature_dim)
    return sorted(valid_ks)

def compute_transfer_retention_ratio(base_drop, ft_drop, epsilon=0.05):
    if base_drop < epsilon:
        return (None, "N/A (Non-Selective in Base)")
    ratio = float(ft_drop / base_drop)
    if ratio >= 0.8:
        category = "Substrate Preservation"
    elif ratio >= 0.2:
        category = "Substrate Reassignment"
    else:
        category = "Substrate Dispersion"
    return (ratio, category)

def compute_per_class_accuracies(predictions, labels, num_classes=6):
    preds, targets = np.asarray(predictions), np.asarray(labels)
    accs = {"overall": float(np.mean(preds == targets)) * 100.0}
    for c in range(num_classes):
        class_mask = targets == c
        class_name = EMOTION_CLASSES[c] if c < len(EMOTION_CLASSES) else f"class_{c}"
        if np.sum(class_mask) == 0:
            accs[class_name] = 0.0
        else:
            accs[class_name] = float(np.mean(preds[class_mask] == targets[class_mask])) * 100.0
    return accs

print("✅ Setup complete and standalone math utilities loaded.")

Runtime Device: cuda:0
✅ Setup complete and standalone math utilities loaded.
